# 08 · Sensitivity analysis: duplicate rows
The main pipeline **drops exact duplicate rows before splitting** (9.5 % of the data). The duplicates are not copy errors: they are different survey respondents with identical answers (the 21 features are almost all binary/ordinal; 67 duplicate groups even have conflicting labels).
Here we check how much the decision matters by running the **alternative protocol: split first, keep duplicates**, and measuring how many test people have an exact twin in train.

In [1]:
import sys; sys.path.insert(0, "..")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns, joblib, shap, warnings
from functools import partialmethod
from tqdm import tqdm; tqdm.__init__ = partialmethod(tqdm.__init__, disable=True)
from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, precision_score, recall_score
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
import lightgbm as lgb
from src.preprocess import get_splits
from src.evaluation import threshold_for_recall
warnings.filterwarnings("ignore"); sns.set_theme(style="whitegrid")

/usr/local/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def fit_models(X_tr, y_tr, X_va, y_va):
    lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(X_tr, y_tr)
    gb = lgb.LGBMClassifier(n_estimators=3000, learning_rate=0.03, num_leaves=15, min_child_samples=100, subsample=0.8, subsample_freq=1,
                            colsample_bytree=0.8, reg_lambda=5.0, random_state=42, verbose=-1)
    gb.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], eval_metric="binary_logloss", callbacks=[lgb.early_stopping(100, verbose=False)])
    return lr, gb

res, models, splits = [], {}, {}
for proto, drop in [("A · drop duplicates (main)", True), ("B · keep duplicates", False)]:
    X_tr, X_va, X_te, y_tr, y_va, y_te = get_splits(drop_duplicates=drop); splits[proto] = (X_tr, X_va, X_te, y_tr, y_va, y_te)
    lr, gb = fit_models(X_tr, y_tr, X_va, y_va); models[proto] = (lr, gb)
    for name, m in [("LogReg", lr), ("LightGBM", gb)]:
        pv, pt = m.predict_proba(X_va)[:, 1], m.predict_proba(X_te)[:, 1]; t = threshold_for_recall(y_va, pv, 0.80)
        res.append({"protocol": proto, "model": name, "n_total": len(X_tr) + len(X_va) + len(X_te), "prevalence": y_te.mean(), "ROC-AUC": roc_auc_score(y_te, pt),
                    "PR-AUC": average_precision_score(y_te, pt), "precision@recall80": precision_score(y_te, pt >= t), "recall": recall_score(y_te, pt >= t)})
pd.DataFrame(res).round(3)

,protocol,model,n_total,prevalence,ROC-AUC,PR-AUC,precision@recall80,recall
0,A · drop duplicates (main),LogReg,229474,0.153,0.807,0.404,0.304,0.797
1,A · drop duplicates (main),LightGBM,229474,0.153,0.818,0.442,0.313,0.799
2,B · keep duplicates,LogReg,253680,0.139,0.821,0.398,0.299,0.802
3,B · keep duplicates,LightGBM,253680,0.139,0.830,0.431,0.304,0.808


## Leakage: how many test people have an exact twin in train?
`twin` = an identical feature vector appears in the training set. If twins are easier to predict, the metrics under protocol B are optimistic.

In [3]:
X_tr, X_va, X_te, y_tr, y_va, y_te = splits["B · keep duplicates"]; gb = models["B · keep duplicates"][1]
h = lambda X: pd.util.hash_pandas_object(X, index=False)
twin = h(X_te).isin(set(h(X_tr))).values
pt = gb.predict_proba(X_te)[:, 1]
print(f"test people with an exact twin in train (protocol B): {twin.mean():.1%}")
tab = pd.DataFrame({"n": [twin.sum(), (~twin).sum()], "prevalence": [y_te.values[twin].mean(), y_te.values[~twin].mean()],
                    "ROC-AUC": [roc_auc_score(y_te.values[twin], pt[twin]), roc_auc_score(y_te.values[~twin], pt[~twin])]}, index=["has twin in train", "no twin"]).round(3)
tab

test people with an exact twin in train (protocol B): 12.6%


,n,prevalence,ROC-AUC
has twin in train,4785,0.051,0.839
no twin,33267,0.152,0.822


## Do the explanations change?
Global mean |SHAP| (2,000 test rows) of the LightGBM under both protocols, 21 features.

In [4]:
imp = {}
for proto, drop in [("A · drop duplicates (main)", True), ("B · keep duplicates", False)]:
    X_tr, X_va, X_te, *_ = splits[proto]; gb = models[proto][1]
    sv = shap.TreeExplainer(gb, data=X_tr.sample(100, random_state=0), feature_perturbation="interventional", model_output="raw")(X_te.sample(2000, random_state=0)).values
    imp[proto] = pd.Series(np.abs(sv).mean(0), index=X_tr.columns)
imp = pd.DataFrame(imp).sort_values("A · drop duplicates (main)", ascending=False)
print("Spearman of global SHAP rankings A vs B:", round(spearmanr(imp.iloc[:, 0], imp.iloc[:, 1])[0], 3))
imp.round(3).head(10)

 50%|==========          | 992/2000 [00:11<00:11]       

 54%|===========         | 1082/2000 [00:12<00:10]       

 59%|============        | 1173/2000 [00:13<00:09]       

 63%|=============       | 1262/2000 [00:14<00:08]       

 67%|=============       | 1346/2000 [00:15<00:07]       

...

 86%|=================   | 1718/2000 [00:16<00:02]       

 91%|==================  | 1824/2000 [00:17<00:01]       

 97%|=================== | 1932/2000 [00:18<00:00]       

Spearman of global SHAP rankings A vs B: 0.942


,A · drop duplicates (main),B · keep duplicates
GenHlth,0.506,0.561
Age,0.387,0.415
BMI,0.368,0.406
HighBP,0.347,0.343
HighChol,0.273,0.284
Sex,0.121,0.134
CholCheck,0.112,0.051
Income,0.112,0.137
MentHlth,0.093,0.057
HeartDiseaseorAttack,0.085,0.056


## Conclusion
- **Keeping duplicates inflates the metrics:** LightGBM ROC-AUC 0.818 (drop duplicates) vs 0.830 (keep); LogReg 0.807 vs 0.821. The two protocols also differ in prevalence (15.3 % vs 13.9 %), so part of the gap is composition.
- **Direct evidence of leakage under protocol B:** 12.6 % of test people (4,785) have an exact twin in train, and those are easier to predict (ROC-AUC 0.839 vs 0.822 for people without a twin).
- **Explanations are similar, not identical:** Spearman 0.94 between the global SHAP rankings; the top 5 features are the same, but some move (e.g. `CholCheck` 0.11 → 0.05, `MentHlth` 0.09 → 0.06).
- We therefore keep protocol A (drop duplicates) as the main one: more conservative, and the conclusions about which features matter do not depend on this choice.